In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import pickle
import os

print("Imports successful!")

Imports successful!


In [2]:
# Reloading data

col_names = ['unit_id', 'cycle', 'op_sett_1', 'op_sett_2', 'op_sett_3'] + \
            [f'sensor_{i}' for i in range(1, 22)]

train_df = pd.read_csv('../data/raw/train_FD001.txt', sep=r'\s+', header=None, names=col_names)
print(train_df.shape)

(20631, 26)


In [3]:
# Dropping the low-variance sensors identified in EDA
low_variance_sensors = ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16',
                        'sensor_18', 'sensor_19', 'sensor_6', 'sensor_15',
                        'sensor_8', 'sensor_13']

train_df.drop(columns=low_variance_sensors, inplace=True)
print(f"Remaining columns: {train_df.shape[1]}")

Remaining columns: 16


We need RUL in this notebook too since it drives the window labeling later.

In [4]:
max_cycles = train_df.groupby('unit_id')['cycle'].max().reset_index()
max_cycles.columns = ['unit_id', 'max_cycle']
train_df = train_df.merge(max_cycles, on='unit_id')
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']
train_df.drop(columns=['max_cycle'], inplace=True)
print(train_df.shape)

(20631, 17)


Per-Engine Normalization
We normalize each engine separately so that differences in operating baseline don't get treated as signal.

In [5]:
sensor_cols = [c for c in train_df.columns 
               if c not in ['unit_id', 'cycle', 'op_sett_1', 'op_sett_2', 'op_sett_3', 'RUL']]

scaler_dict = {}
normalized_frames = []

for engine_id, group in train_df.groupby('unit_id'):
    scaler = MinMaxScaler()
    group = group.copy()
    group[sensor_cols] = scaler.fit_transform(group[sensor_cols])
    scaler_dict[engine_id] = scaler
    normalized_frames.append(group)

train_norm = pd.concat(normalized_frames).reset_index(drop=True)
print(f"Normalized shape: {train_norm.shape}")
print(train_norm[sensor_cols].describe().round(3))

Normalized shape: (20631, 17)
        sensor_2   sensor_3   sensor_4   sensor_7   sensor_9  sensor_11  \
count  20631.000  20631.000  20631.000  20631.000  20631.000  20631.000   
mean       0.421      0.435      0.382      0.607      0.393      0.364   
std        0.194      0.194      0.203      0.201      0.242      0.206   
min        0.000      0.000      0.000      0.000      0.000      0.000   
25%        0.284      0.298      0.234      0.491      0.177      0.216   
50%        0.397      0.417      0.345      0.641      0.379      0.317   
75%        0.540      0.552      0.497      0.751      0.577      0.474   
max        1.000      1.000      1.000      1.000      1.000      1.000   

       sensor_12  sensor_14  sensor_17  sensor_20  sensor_21  
count  20631.000  20631.000  20631.000  20631.000  20631.000  
mean       0.623      0.415      0.421      0.591      0.590  
std        0.206      0.260      0.205      0.197      0.198  
min        0.000      0.000      0.000    

We're also saving each engine's scaler in scaler_dict — you'll need those later to inverse-transform predictions.

Building rolling windows for the LSTM autoencoder.

In [6]:
window_size = 30
failure_threshold = 30

feature_cols = sensor_cols

def create_sequences(df, window_size):
    sequences = []
    labels = []
    unit_ids = []
    rul_targets = []
    
    for engine_id, group in df.groupby('unit_id'):
        group = group.sort_values('cycle').reset_index(drop=True)
        values = group[feature_cols].values
        rul = group['RUL'].values
        
        for i in range(len(group) - window_size + 1):
            seq = values[i:i+window_size]
            seq_rul = rul[i+window_size-1]
            label = 1 if seq_rul < failure_threshold else 0
            sequences.append(seq)
            labels.append(label)
            unit_ids.append(engine_id)
            rul_targets.append(seq_rul)
            
    return np.array(sequences), np.array(labels), np.array(unit_ids), np.array(rul_targets)

X, y, engine_ids, rul_targets = create_sequences(train_norm, window_size)

print(X.shape, y.shape)
print("Anomaly rate:", y.mean().round(4))

(17731, 30, 11) (17731,)
Anomaly rate: 0.1692


Each window becomes one training example, and the label marks whether the engine was already in the failure zone at the end of that window. That gives you the supervised target for evaluating anomaly detection later, even though the autoencoder itself will still train on normal behavior.

In [7]:
# Saving processed arrays so the modeling notebook can load them directly
os.makedirs('../data/processed', exist_ok=True)

np.save('../data/processed/X_windows.npy', X)
np.save('../data/processed/y_labels.npy', y)
np.save('../data/processed/engine_ids.npy', engine_ids)
np.save('../data/processed/rul_targets.npy', rul_targets)

with open('../data/processed/scaler_dict.pkl', 'wb') as f:
    pickle.dump(scaler_dict, f)

print("Saved processed files!")

Saved processed files!


In [8]:
#Personal sanity check
print("X sample shape:", X[0].shape)
print("First label:", y[0])
print("First RUL target:", rul_targets[0])

X sample shape: (30, 11)
First label: 0
First RUL target: 162


In [9]:
print("Total windows:", len(X))
print("Normal windows:", (y == 0).sum())
print("Anomalous windows:", (y == 1).sum())
print("Feature dimensions:", X.shape[1], "timesteps by", X.shape[2], "features")

Total windows: 17731
Normal windows: 14731
Anomalous windows: 3000
Feature dimensions: 30 timesteps by 11 features
